# Embeddings por artículo — Qwen3-Embedding-4B

**Entregable**

- `qwen3_4b_abstract.npz`: un vector por **abstract** (versión principal).
- `qwen3_4b_titulo_keywords_abstract.npz`: un vector por **título + keywords + abstract** (versión opcional).

Ambos contienen la matriz `vectores` de **519 × 2560** y `article_ids` en el mismo orden de las filas. También conservan el modelo, revisión y huella SHA-256 del catálogo.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import torch

MODELO = "Qwen/Qwen3-Embedding-4B"
REVISION = "5cf2132abc99cad020ac570b19d031efec650f2b"
LOTE = 4


REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "06_grafos/catalogo_articulos.csv").is_file()
             and (p / "08_Embedding").is_dir()), None)
if REPO is None:
    raise FileNotFoundError("No se encuentra el repositorio ni 06_grafos/catalogo_articulos.csv")

ENTRADA = REPO / "06_grafos/catalogo_articulos.csv"
SALIDA = REPO / "08_Embedding"

# Preserva article_id como texto y el orden original del catálogo.
df = pd.read_csv(ENTRADA, encoding="utf-8-sig", dtype={"article_id": "string"}, keep_default_na=False)
columnas = {"article_id", "titulo", "keywords", "abstract"}
if not columnas.issubset(df.columns):
    raise ValueError(f"Faltan columnas: {sorted(columnas - set(df.columns))}")

ids = df["article_id"].astype(str).str.strip().tolist()
if any(not x for x in ids) or len(ids) != len(set(ids)):
    raise ValueError("Hay article_id vacíos o duplicados")
if df["abstract"].astype(str).str.strip().eq("").any():
    raise ValueError("Hay abstracts vacíos")

huella_catalogo = hashlib.sha256(ENTRADA.read_bytes()).hexdigest()
print(f"Artículos: {len(df)} | Modelo: {MODELO}")
print("Entrada:", ENTRADA)
print("Salida:", SALIDA)


Artículos: 519 | Modelo: Qwen/Qwen3-Embedding-4B
Entrada: c:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\06_grafos\catalogo_articulos.csv
Salida: c:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\08_Embedding


In [2]:
# Se respeta el texto del abstract. En la opción ampliada se omiten keywords vacías.
textos_abstract = df["abstract"].astype(str).tolist()

def texto_ampliado(fila):
    partes = [f"Title: {fila['titulo']}"]
    if str(fila["keywords"]).strip():
        partes.append(f"Keywords: {fila['keywords']}")
    partes.append(f"Abstract: {fila['abstract']}")
    return "\n".join(partes)

textos_ampliados = [texto_ampliado(fila) for _, fila in df.iterrows()]

versiones = {
    "abstract": textos_abstract,
    "titulo_keywords_abstract": textos_ampliados,
}
archivos = {nombre: SALIDA / f"qwen3_4b_{nombre}.npz" for nombre in versiones}

# Si alguna versión ya se guardó, comprueba su integridad y no la recalcula.
pendientes = {}
for nombre, textos in versiones.items():
    ruta = archivos[nombre]
    if ruta.exists():
        with np.load(ruta, allow_pickle=False) as previo:
            if (previo["vectores"].shape != (len(ids), 2560)
                    or previo["article_ids"].tolist() != ids
                    or previo["revision"].item() != REVISION
                    or previo["campos"].item() != nombre
                    or previo["sha256_catalogo"].item() != huella_catalogo):
                raise ValueError(f"El archivo existente no coincide con el catálogo: {ruta.name}")
        print("Ya generado (se conserva):", ruta.name)
    else:
        pendientes[nombre] = textos

print("Versiones pendientes:", list(pendientes))


Versiones pendientes: ['abstract', 'titulo_keywords_abstract']


In [3]:
if not pendientes:
    print("Las dos versiones ya están guardadas; no hay que cargar el modelo otra vez.")
else:
    from sentence_transformers import SentenceTransformer

    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError("Se necesita CUDA con soporte bfloat16; usa el entorno de tesis y tu RTX 4070 SUPER")

    # Se conserva la misma revisión y precisión utilizadas en la evaluación.
    modelo = SentenceTransformer(
        MODELO, revision=REVISION, device="cuda",
        model_kwargs={"torch_dtype": torch.bfloat16},
    )
    modelo.eval()
    if modelo.get_sentence_embedding_dimension() != 2560:
        raise ValueError("La dimensión del modelo no es 2560")
    if modelo.default_prompt_name is not None:
        raise ValueError("El modelo tiene un prefijo predeterminado inesperado para documentos")

    for nombre, textos in pendientes.items():
        # Verificar que ningún texto se trunque silenciosamente.
        tokens = modelo.tokenizer(textos, truncation=False, padding=False)["input_ids"]
        max_tokens = max(map(len, tokens))
        if max_tokens > min(int(modelo.max_seq_length), 32768):
            raise ValueError(f"{nombre}: hay un texto demasiado largo ({max_tokens} tokens)")

        print(f"Generando {nombre} ({len(textos)} artículos)...")
        vectores = modelo.encode(textos, batch_size=LOTE, show_progress_bar=True,
                                normalize_embeddings=True, convert_to_numpy=True)
        vectores = np.asarray(vectores, dtype=np.float32)
        normas = np.linalg.norm(vectores, axis=1, keepdims=True)
        if vectores.shape != (len(ids), 2560) or not np.isfinite(vectores).all() or np.any(normas == 0):
            raise ValueError(f"Vectores inválidos para {nombre}")
        vectores /= normas  # Normalización L2 en float32, lista para similitud coseno.

        # Un solo archivo por variante; metadatos incluidos dentro del mismo NPZ.
        with archivos[nombre].open("xb") as f:
            np.savez_compressed(
                f, vectores=vectores, article_ids=np.asarray(ids),
                modelo=np.asarray(MODELO), revision=np.asarray(REVISION),
                campos=np.asarray(nombre), normalizacion=np.asarray("L2"),
                sha256_catalogo=np.asarray(huella_catalogo),
                max_tokens_texto=np.asarray(max_tokens),
            )
        print("Guardado:", archivos[nombre].name, "| forma:", vectores.shape)

    print("Listo: dos archivos NPZ en 08_Embedding.")


W1110 05:25:31.665000 19344 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Generando abstract (519 artículos)...


C:\Users\hazar\AppData\Local\Temp\ipykernel_19344\1941347667.py:15: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  if modelo.get_sentence_embedding_dimension() != 2560:


Batches:   0%|          | 0/130 [00:00<?, ?it/s]

Guardado: qwen3_4b_abstract.npz | forma: (519, 2560)
Generando titulo_keywords_abstract (519 artículos)...


Batches:   0%|          | 0/130 [00:00<?, ?it/s]

Guardado: qwen3_4b_titulo_keywords_abstract.npz | forma: (519, 2560)
Listo: dos archivos NPZ en 08_Embedding.


In [4]:
# Verificación final: las filas de vectores corresponden a los mismos article_id.
for nombre, ruta in archivos.items():
    with np.load(ruta, allow_pickle=False) as datos:
        matriz = datos["vectores"]
        if (matriz.shape != (len(ids), 2560)
                or datos["article_ids"].tolist() != ids
                or datos["sha256_catalogo"].item() != huella_catalogo
                or not np.isfinite(matriz).all()
                or not np.allclose(np.linalg.norm(matriz, axis=1), 1, atol=1e-5)):
            raise ValueError(f"La validación falló: {ruta.name}")
        print(f"OK: {ruta.name} | {matriz.shape[0]} artículos × {matriz.shape[1]} dimensiones")


OK: qwen3_4b_abstract.npz | 519 artículos × 2560 dimensiones
OK: qwen3_4b_titulo_keywords_abstract.npz | 519 artículos × 2560 dimensiones
